<a href="https://colab.research.google.com/github/xzhen-hub/ML_Linear-Factor-Model/blob/main/g1_freddie_full61_20261102.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Freddie: original October payoff tree versus a new expanded Freddie tree
Prepared November 2, 2026 UTC. Run in Colab, CPU runtime; save the notebook to Google Drive. This is a new experiment, not a claim that a Freddie expanded model already exists.

## What was verified first
The read-only production registry shows **one conventional 30YR tree family trained on Freddie AND Fannie**, with agency-specific calibration. Its payoff head is `conv_prepay_30y_v2_payoff_prod_t360_v2_20261002`. Freddie's separate served surface is `scurve-v2-surface-20260901e2s3-freddie30`, whose `source_model_version` is `v2-20261002`. The historical version is pinned even though that surface is now deprecated in favor of its SOFR successor.

Registry parameters: Freddie payoff factor **0.1185**, curtailment factor **0.0732**, conditional curtailment magnitude **0.068599269**. Fannie differs: 0.1174, 0.0711, 0.060618837. The old final served surfaces are agency-specific; the underlying payoff tree is shared. Do not run the Fannie surface or Fannie calibration on Freddie.

## Years, windows and scope
- **Native Freddie fitting/calendar window: May2021–May2026 inclusive = 61 months.** Registry's `2021-05-01..2026-06-01` uses an exclusive end. Do not add 2020 in this primary test.
- A second report uses January2022–May2026 = 53 calendar months to align the Fannie comparison period. It does not shorten Freddie training.
- Keep the problematic **2024 collateral-vintage** and 6.0/6.5 actual pool-coupon buckets. Training uses broad Freddie histories, not only 2024 G1 loans.
- The included research roster follows the previous Fannie research convention: issue date minus explicit issuance WALA. There are 12,130 verified target pools, first issued January2024, plus unresolved candidate pools listed separately. Thus early calendar rows legitimately have no target loans.
- **This roster is not certified official DPR constituent membership.** The owner's serving key uses first-payment year and FLOOR note-rate conventions; DPR is a published voluntary pool benchmark. Issue year, adjusted collateral year and first-payment year are distinct. We do not substitute one for another silently. This notebook measures the same kind of actual-pool, full-payoff research comparison as the completed Fannie test, not the final served DPR CPR miss.

## Why a new Freddie fitting step is necessary
The saved expansion archive contains a Fannie expanded model and Freddie SMALL research models; no Freddie expanded model exists there. This notebook verifies/replays the saved Freddie 25-feature SMALL baseline, then expands the same original approximately 5% loan-hash parent. Same seed, exact matrix, frozen Freddie encoder, uniform training weights, labels, 12% negative retention, loan-ID split and original validation set. All held-out loan IDs remain excluded from fitting across every month. Newly added held-out loans are not added to the stopping set. The original small model is a replay check, NOT the original production comparator.

Primary result: October production tree vs newly fitted Freddie expanded tree, both with the same 0.12 negative-sampling odds correction, before additional calibration. This mirrors the Fannie test. It does not isolate splitting: agency mix, engine, stopping and sampling realization also differ from production.

## Upload TWO ZIPs using the Colab Files sidebar
1. `g1_sample_expansion_uxtgy0f6_FULL_recovery.zip` from `Y:/OASIVE` (about192MB). NOT the Fannie full53/monthly ZIP and NOT a small-results ZIP.
2. `g1_freddie_full61_inputs_20261102.zip`, provided with this notebook (about0.5MB): helpers, registry evidence, pool metadata, source contracts.

Wait for readable ZIPs. All BigQuery writes are new tables in `oasive_quant_sandbox`. No bucket, registry change, production trainer or deployment. Training happens locally in Colab; allow adequate CPU time and RAM. Do not reduce data after inspecting outcomes to pass a memory check.


## Step 1 — Restore exact inputs (or this notebook's FULL checkpoint)
For a fresh start, leave `RESTORE_ZIP=None`. For runtime recovery, upload this notebook's own FULL checkpoint and set its exact path. Recovery preserves original inputs and checksums; it never retrains automatically.


In [ ]:
from pathlib import Path
import json,hashlib,zipfile,shutil,sys,subprocess,importlib,importlib.util,gc,uuid
ROOT=Path('/content/g1_freddie_full61_20261102');ROOT.mkdir(exist_ok=True)
ORIGINAL_FULL=Path('/content/g1_sample_expansion_uxtgy0f6_FULL_recovery.zip')
TOOLS=Path('/content/g1_freddie_full61_inputs_20261102.zip')
RESTORE_ZIP=None

def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def restore(archive,destination,manifest_name,selected=None):
    archive=Path(archive);destination=Path(destination)
    assert archive.is_file() and zipfile.is_zipfile(archive),f'Missing/incomplete ZIP: {archive}'
    with zipfile.ZipFile(archive) as z:
        hashes=json.loads(z.read(manifest_name));names=list(hashes) if selected is None else selected
        for name in names:
            assert name in hashes,'Wrong ZIP; missing '+name
            p=(destination/name).resolve();assert p.is_relative_to(destination.resolve())
            if not p.exists():
                p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_suffix(p.suffix+'.recovering')
                with z.open(name) as src,tmp.open('wb') as dst:shutil.copyfileobj(src,dst,1024*1024)
                assert sha(tmp)==hashes[name],'Corrupt archive member: '+name
                tmp.replace(p)
            assert sha(p)==hashes[name],'Conflicting recovered file: '+name
        return {n:hashes[n] for n in names}
if RESTORE_ZIP:restore(RESTORE_ZIP,ROOT,'freddie_recovery_hashes.json')
if not (ROOT/'tools/freddie_experiment_contract.json').exists():
    assert sha(TOOLS)=='2a093de82e676d83503d5ef486e3450db2cef252533ff53e2135487cf1c0bfe0','Wrong tools ZIP'
    restore(TOOLS,ROOT/'tools','freddie_input_hashes.json')
selected=['original_saved_experiment/manifest.json','original_saved_experiment/post_rally_matched_test.py',
          'original_saved_experiment/models_20261021/freddie_baseline.ubj',
          'original_saved_experiment/models_20261021/freddie_encoder.joblib',
          'original_saved_experiment/training_month_split_freddie.csv']
lock=ROOT/'original_input_hashes.json'
if not lock.exists():
    hashes=restore(ORIGINAL_FULL,ROOT/'original','full_recovery_file_hashes.json',selected)
    lock.write_text(json.dumps(hashes,indent=2))
for n,h in json.loads(lock.read_text()).items():assert sha(ROOT/'original'/n)==h
ORIG=ROOT/'original/original_saved_experiment'
MANIFEST=json.loads((ORIG/'manifest.json').read_text())
CONTRACT=json.loads((ROOT/'tools/freddie_experiment_contract.json').read_text())
print('Recovered saved Freddie SMALL baseline/encoder, exact module and owner evidence.')


Recovered saved Freddie SMALL baseline/encoder, exact module and owner evidence.


## Step 2 — Restore package versions
If installation occurs, restart the session, then rerun Steps1–2. Keep recorded versions; do not change model representation to solve an installation issue.


In [ ]:
import importlib.metadata
requirements=[]
for name,version in MANIFEST['versions'].items():
    try:current=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=version:requirements.append(name+'=='+version)
if requirements:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*requirements],check=True)
    raise RuntimeError('Restart session, rerun Steps1–2, then continue.')
print('Recorded versions match.')


Recorded versions match.


## Step 3 — Load the exact Freddie baseline and research helpers
No fitting yet. The SMALL model is only a reproducibility control. The separate October production tree is loaded from BigQuery at inference time.


In [ ]:
import numpy as np,pandas as pd,joblib,xgboost as xgb

def load(name,path):
    spec=importlib.util.spec_from_file_location(name,str(path));mod=importlib.util.module_from_spec(spec)
    sys.modules[name]=mod;spec.loader.exec_module(mod);return mod
core=load('g1_saved_payoff_comparison_core',ROOT/'tools/g1_saved_payoff_comparison_core.py')
expansion=load('g1_freddie_sample_expansion',ROOT/'tools/g1_freddie_sample_expansion.py')
full=load('g1_freddie_full61',ROOT/'tools/g1_freddie_full61.py')
m=load('exact_original_matrix',ORIG/'post_rally_matched_test.py')
assert sha(m.__file__)==MANIFEST['code_sha256'] and m.BASE==core.BASE and m.CAT==core.CAT
assert m.Q==.12 and m.PARAMS==MANIFEST['params']
assert expansion.START=='2021-05-01' and expansion.END=='2026-05-01' and len(full.MONTHS)==61
ENCODER=joblib.load(ORIG/'models_20261021/freddie_encoder.joblib')
SMALL=xgb.Booster(model_file=str(ORIG/'models_20261021/freddie_baseline.ubj'))
SMALL.set_param({'nthread':2,'device':'cpu'})
CONFIG=dict(seed=20261021,expanded_parent_bucket_max=10000,new_timing_features=False,loss_weights='one')
expansion.validate_spec(m,MANIFEST,CONFIG)
print('Freddie SMALL saved best iteration:',SMALL.best_iteration)


Freddie SMALL saved best iteration: 69


## Step 4 — Authenticate and pin training sources and October model
Read-only source access. The frozen parent is already the original approximately5% loan sample. Expansion removes only the later nested20% restriction. It does not create more payoff events by retaining extra negatives alone.


In [ ]:
from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()
CLIENT=bigquery.Client(project=core.PROJECT,location='us-central1')
SOURCE_CONTRACT=json.loads((ROOT/'tools/freddie_training_source_contract.json').read_text())
PINNED=expansion.pin_sources(CLIENT,SOURCE_CONTRACT,MANIFEST)
PROD_ID=core.model_id(CLIENT.get_model(full.MODEL))
OWNER_META=json.loads((ROOT/'tools/production_payoff_metadata.json').read_text())
assert PROD_ID==core.model_id_from_dict(OWNER_META),'October production artifact changed'
EXP_WORK=ROOT/'training';EXP_WORK.mkdir(exist_ok=True)
plan=dict(agency='freddie',training_start=expansion.START,training_last_month=expansion.END,config=CONFIG,
          source_identities=PINNED,production_identity=PROD_ID,expansion_code_sha256=sha(expansion.__file__),matrix_sha256=sha(m.__file__))
p=EXP_WORK/'training_plan.json'
if p.exists():assert json.loads(p.read_text())==plan,'Changed training plan: use a new run root'
else:core.write_json(p,plan)
print('Native training window May2021–May2026; original saved stopping set remains fixed.')


Native training window May2021–May2026; original saved stopping set remains fixed.


## Step 5 — Dry-run training source audits and extraction
Review estimates before the actual SELECTs in Step6. Caps do not mean the entire cap will be billed. Source tables stay unchanged. The audit verifies original rows/features/labels/weights/retention before any model is trained.


In [ ]:
TRAIN_QUERIES={'source_audit':expansion.audit_sql(m,CONFIG),'source_support':expansion.support_sql(m,CONFIG),
               'retained_parent':expansion.extract_sql(m,CONFIG)}
TRAIN_ESTIMATES=[]
for name,q in TRAIN_QUERIES.items():
    dry=expansion.dry_run(CLIENT,q,full.CAP)
    (EXP_WORK/(name+'.sql')).write_text(q)
    TRAIN_ESTIMATES.append(dict(query=name,**dry))
TRAIN_ESTIMATES=pd.DataFrame(TRAIN_ESTIMATES);TRAIN_ESTIMATES.to_csv(EXP_WORK/'training_read_estimates.csv',index=False)
assert TRAIN_ESTIMATES.estimated_bytes.sum()<=full.TOTAL_CAP
print(TRAIN_ESTIMATES.to_string(index=False))


          query  estimated_bytes  estimated_gib                                                       sql_sha256
   source_audit      20557002649      19.145201 6fefe480b60ded246e47a99b5bc5e846f54dde0319d1afff2e7c6d6063928e2b
 source_support       6982647641       6.503097 b72d1882247e02d08d4a3316b934253b55c94ec7804035b02eb10ee30317b85b
retained_parent      17062796258      15.890967 d3d01648d472c37453cf913653a6665339d5babff89d92aba4df013cfbbd1429


## Step 6 — Audit fitting support across all61 months
Actual queries. Review payoff counts, retained negatives, month coverage, state/property/DTI representation. Training weights remain one. Evaluation group counts are not substitutes for fitting support. Do not train only the large G1 loans known to be difficult.


In [ ]:
AUDIT=expansion.small_query(CLIENT,TRAIN_QUERIES['source_audit'],full.CAP,EXP_WORK,'source_audit')
AUDIT=expansion.check_audit(AUDIT)
SUPPORT=expansion.small_query(CLIENT,TRAIN_QUERIES['source_support'],full.CAP,EXP_WORK,'source_support')
display(AUDIT)
display(SUPPORT.loc[SUPPORT.dimension.isin(['focus_profile','payoff_component'])])
print('All source reconciliation checks passed. Review saved source_support.csv before fitting.')


source_audit dry-run GiB: 19.145
source_audit query job: 60fc597a-0a62-4fd8-86b0-65667be7f4a6
source_support dry-run GiB: 6.503
source_support query job: dd45edd1-f8fe-4153-a396-e2d6897ee68f


,month,is_original,parent_rows,distinct_parent_rows,missing_parent_original_rows,missing_original_rows,unexpected_original_rows,bad_history_rows,bad_upb_rows,history_value_conflicts,label_rule_conflicts,original_feature_conflicts,original_label_weight_retention_conflicts,payoffs,retained_rows,retained_payoffs,retained_nonpayoffs,upb_month_exposure
0,2021-05-01,False,152250,152250,0,0,0,0,0,0,0,0,0,2106,20247,2106,18141,4.330690e+10
1,2021-05-01,True,37943,37943,0,0,0,0,0,0,0,0,0,501,5027,501,4526,1.077191e+10
2,2021-06-01,False,164419,164419,0,0,0,0,0,0,0,0,0,2432,21697,2432,19265,4.694242e+10
3,2021-06-01,True,40926,40926,0,0,0,0,0,0,0,0,0,640,5553,640,4913,1.165855e+10
4,2021-07-01,False,170969,170969,0,0,0,0,0,0,0,0,0,2404,22713,2404,20309,4.885395e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,2026-03-01,True,80855,80855,0,0,0,0,0,0,0,0,0,728,10418,728,9690,2.299468e+10
118,2026-04-01,False,324886,324886,0,0,0,0,0,0,0,0,0,2600,41160,2600,38560,9.195931e+10
119,2026-04-01,True,80715,80715,0,0,0,0,0,0,0,0,0,601,10212,601,9611,2.290553e+10
120,2026-05-01,False,325305,325305,0,0,0,0,0,0,0,0,0,2318,41080,2318,38762,9.206812e+10


,month,is_original,dimension,bucket,row_count,payoffs,retained_rows,retained_payoffs,retained_nonpayoffs,upb_month_exposure
14,2021-05-01,False,focus_profile,large|0.5_1,668,42,110,42,68,3.797507e+08
15,2021-05-01,False,focus_profile,large|1_1.5,316,15,53,15,38,1.821745e+08
16,2021-05-01,False,focus_profile,medium|0.5_1,8290,198,1177,198,979,1.759173e+09
17,2021-05-01,False,focus_profile,medium|1_1.5,5584,194,829,194,635,1.150572e+09
18,2021-05-01,False,focus_profile,other,137392,1657,18078,1657,16421,3.983523e+10
...,...,...,...,...,...,...,...,...,...,...
9687,2026-05-01,True,focus_profile,medium|1_1.5,214,6,27,6,21,4.458431e+07
9688,2026-05-01,True,focus_profile,other,79472,576,10096,576,9520,2.246448e+10
9689,2026-05-01,True,payoff_component,<no_component>,80263,0,9690,0,9690,2.277029e+10
9690,2026-05-01,True,payoff_component,buyout,40,40,40,40,0,1.026529e+07


All source reconciliation checks passed. Review saved source_support.csv before fitting.


## Step 7 — Download retained rows in resumable shards
Keep all payoff rows and the same deterministic12% negative sample within the5% parent. This queries/downloads broad Freddie histories, not only the G1 cohort. No BigQuery training table/model is created.


In [ ]:
expansion.stream_extract(CLIENT,TRAIN_QUERIES['retained_parent'],full.CAP,EXP_WORK,PINNED,m)


retained_parent dry-run GiB: 15.891
retained_parent query job: 8beaaaba-7e8b-49b7-b093-65091d194fbf
Retained rows downloaded: 102940
Retained rows downloaded: 205880
Retained rows downloaded: 308820
Retained rows downloaded: 411760
Retained rows downloaded: 514700
Retained rows downloaded: 617640
Retained rows downloaded: 720580
Retained rows downloaded: 823520
Retained rows downloaded: 926460
Retained rows downloaded: 1029400
Retained rows downloaded: 1132340
Retained rows downloaded: 1235280
Retained rows downloaded: 1338220
Retained rows downloaded: 1441160
Retained rows downloaded: 1544100
Retained rows downloaded: 1647040
Retained rows downloaded: 1749980
Retained rows downloaded: 1852920
Retained rows downloaded: 1955860
Retained rows downloaded: 2058800
Retained rows downloaded: 2161740
Retained rows downloaded: 2264680
Retained rows downloaded: 2367620
Retained rows downloaded: 2470560
Retained rows downloaded: 2573500
Retained parent download complete: 2592531 rows


[{'file': 'part_00000.parquet',
  'rows': 10294,
  'sha256': '4fe1662c5a52b5905ec6bb1c4173a257bca7790f63ae552d286c430bd99fc0ea'},
 {'file': 'part_00001.parquet',
  'rows': 10294,
  'sha256': '0cf9b254cb6ee1cf40c3d0bc809d73f71783d8ceb67dd73bc736efebcc68737b'},
 {'file': 'part_00002.parquet',
  'rows': 10294,
  'sha256': 'b8f710338b071869bb7ab620df9145c9ad8bac25f2b0b002fc54781016464e8c'},
 {'file': 'part_00003.parquet',
  'rows': 10294,
  'sha256': 'dda2de44706a674ecc0b58fbf7bc4c3c96f68273b523c67cae36d2fc5a7768f3'},
 {'file': 'part_00004.parquet',
  'rows': 10294,
  'sha256': 'dd06dfda2f4245c3f2948487768e1fb09a875e7b1f413aa5dcd5f3a07868d316'},
 {'file': 'part_00005.parquet',
  'rows': 10294,
  'sha256': '8cbda10b117e5a782702723ee600e0aecf4b350b69ef50ba2e0b71b9c5e34511'},
 {'file': 'part_00006.parquet',
  'rows': 10294,
  'sha256': '9b1d2dd60c29bca1175dce337016431985da51b34621b6c99a2eba8dfc849bd8'},
 {'file': 'part_00007.parquet',
  'rows': 10294,
  'sha256': '647b93d8a8b6d71d9373024108b4

## Step 8 — Check RAM, restore the exact split and inspect support
The memory guard is the same conservative75% of available RAM. If it fails, use more RAM or predeclare a different experiment before outcomes; do not disable the guard.

The original small Freddie population must reproduce **438832 fit rows and77784 validation rows**. The expanded fitting set includes newly added fitting loans; all loan-ID validation loans remain excluded across every month. Only the original stopping rows decide when training stops.


In [ ]:
if (ROOT/'expanded_freddie_25features_seed20261021/fit_complete.json').exists():
    print('Completed fit exists; skip reloading training matrices. Step9 verifies and restores it.')
    display(pd.read_csv(EXP_WORK/'fit_validation_population_counts.csv'))
else:
    MEMORY=expansion.memory_plan(EXP_WORK,m);core.write_json(EXP_WORK/'memory_plan.json',MEMORY);print(json.dumps(MEMORY,indent=2))
    assert MEMORY['conservative_memory_check_passed'],'Use adequate RAM; do not change design after outcomes'
    POP,COUNTS=expansion.load_population(EXP_WORK,m,MANIFEST,AUDIT)
    PROFILE,PROFILE_DETAIL=expansion.summarize_support(POP,EXP_WORK)
    display(COUNTS);display(PROFILE)
    print('Detailed state/property/DTI fitting support saved.')


Categorical dictionary shards checked: 40
Categorical dictionary shards checked: 80
Categorical dictionary shards checked: 120
Categorical dictionary shards checked: 160
Categorical dictionary shards checked: 200
Categorical dictionary shards checked: 240
{
  "retained_rows": 2592531,
  "approx_peak_gib": 4.514873098049527,
  "available_gib": 10.804637908935547,
  "conservative_memory_check_passed": true,
  "storage_optimization": "shared categorical dictionaries; no source/sample/feature-value change",
  "sample_original_bytes_per_row": 550.7794831940937,
  "sample_compact_bytes_per_row": 167.2444142218768,
  "note": "Heuristic, not a memory guarantee. Keep the same 75% RAM guard and all experiment settings."
}


,population,rows,unique_loans,payoffs,unique_payoff_loans,months
0,fit_original,438832,85439,21924,21924,61
1,fit_expanded,2204366,429319,109936,109936,61
2,common_validation,77784,15180,3877,3877,61
3,supplemental_validation,310381,60249,15442,15442,61


,arm,incentive_band,balance_role,rows,unique_loans,payoffs,unique_payoff_loans,months
0,original,0.5_1,large,1009,798,240,240,36
1,original,0.5_1,medium,3966,3030,593,593,42
2,original,1_1.5,large,403,303,104,104,33
3,original,1_1.5,medium,1735,1404,368,368,37
4,expanded,0.5_1,large,5071,3895,1160,1160,41
5,expanded,0.5_1,medium,19681,15030,2915,2915,42
6,expanded,1_1.5,large,1956,1526,535,535,35
7,expanded,1_1.5,medium,8964,7186,1881,1881,39


Detailed state/property/DTI fitting support saved.


## Step 9 — Fit the Freddie expanded model, with a replay check first
This is the only fitting step. Set `RUN_FIT=True` after source/support review, then run. First the small baseline is refitted with unchanged settings and checked against the saved Freddie SMALL model. Only if it reproduces does expanded fitting proceed. Replay uses the common validation rows, not a fabricated Freddie March G1 golden file.

Never call the small local control the original production model. The later headline comparator is the exact October BigQuery artifact. If a partial fit directory exists after failure, preserve/export it and use a deliberately new fit directory; do not silently overwrite or skip the replay.


In [ ]:
RUN_FIT=True  # Change to True after reviewing Steps6 and8.
FIT_DIR=ROOT/'expanded_freddie_25features_seed20261021'
if (FIT_DIR/'fit_complete.json').exists():
    fit_record=json.loads((FIT_DIR/'fit_complete.json').read_text())
    assert fit_record['training_plan_sha256']==sha(EXP_WORK/'training_plan.json')
    for name,h in fit_record['files'].items():assert sha(FIT_DIR/name)==h
    print('Reusing completed verified Freddie expanded fit.')
else:
    assert RUN_FIT,'Set RUN_FIT=True after source/support review; no models fitted yet.'
    assert not FIT_DIR.exists(),'Partial fit exists; preserve it and use a new named FIT_DIR'
    replay_rows=POP.loc[POP.common_validation].copy()
    models,FIT_AUDIT=expansion.fit_expansion(POP,m,MANIFEST,ENCODER,SMALL,replay_rows,FIT_DIR,CONFIG)
    files={p.name:sha(p) for p in FIT_DIR.iterdir() if p.is_file()}
    core.write_json(FIT_DIR/'fit_complete.json',dict(training_plan_sha256=sha(EXP_WORK/'training_plan.json'),files=files))
EXPANDED=xgb.Booster(model_file=str(FIT_DIR/'freddie_expanded_25features.ubj'));EXPANDED.set_param({'nthread':2,'device':'cpu'})
ENCODER=joblib.load(FIT_DIR/'fixed_freddie_encoder.joblib')
display(pd.read_csv(FIT_DIR/'fit_audit.csv'))
# Free training matrices before full-population scoring.
if 'POP' in globals():del POP
if 'replay_rows' in globals():del replay_rows
gc.collect()


[0]	validation-logloss:0.19292
[10]	validation-logloss:0.18065
[20]	validation-logloss:0.17857
[30]	validation-logloss:0.17805
[40]	validation-logloss:0.17790
[50]	validation-logloss:0.17785
[60]	validation-logloss:0.17780
[70]	validation-logloss:0.17774
[80]	validation-logloss:0.17775
[88]	validation-logloss:0.17776
[0]	validation-logloss:0.19274
[10]	validation-logloss:0.18032
[20]	validation-logloss:0.17810
[30]	validation-logloss:0.17742
[40]	validation-logloss:0.17714
[50]	validation-logloss:0.17694
[60]	validation-logloss:0.17687
[70]	validation-logloss:0.17679
[80]	validation-logloss:0.17671
[90]	validation-logloss:0.17666
[100]	validation-logloss:0.17661
[110]	validation-logloss:0.17658
[120]	validation-logloss:0.17655
[130]	validation-logloss:0.17655
[138]	validation-logloss:0.17657


,arm,fit_rows,fit_payoffs,common_validation_rows,best_iteration,near_round_limit
0,original,438832,21924,77784,69,False
1,expanded,2204366,109936,77784,119,False


7

## Step 10 — Inspect and upload the versioned Freddie research pool roster
This is a verified-metadata SUBSET experiment, not certified official DPR constituent coverage. First issue is January2024, so early months will be empty. Missing issuance metadata is not assumed to mean2024 or another year. Step12 measures uncertain premium exposure separately. No Fannie pool table is reused.

This creates ONE uniquely named roster table in your sandbox with WRITE_EMPTY. The later matched panel is the only other permanent BigQuery write. Both expire after30days.


In [ ]:
POOL=pd.read_csv(ROOT/'tools/freddie_research_pool_roster.csv',dtype={'pool_id':str})
assert not POOL.pool_id.duplicated().any() and POOL.agency.eq('freddie').all()
assert int(POOL.mapping_status.eq('known_target').sum())==CONTRACT['target_pools']
target=POOL.loc[POOL.mapping_status.eq('known_target')]
assert target.pool_vintage.eq(2024).all() and target.product_term.eq('30YR').all()
print('Known target pools:',len(target),'Earliest issue:',target.issue_date.min())
print('Unresolved candidate pools:',int(POOL.mapping_status.eq('missing_issuance_metadata').sum()))
EVAL=ROOT/'evaluation';EVAL.mkdir(exist_ok=True)
prep_file=EVAL/'preparation_spec.json'
if prep_file.exists():PREP=json.loads(prep_file.read_text())
else:
    suffix=uuid.uuid4().hex[:12]
    sources=[full.FEATURES,full.LABELS,full.FACTS,full.DIM]
    PREP=dict(panel=full.PROJECT+'.oasive_quant_sandbox.g1_freddie_full61_'+suffix+'_panel',
              roster=full.PROJECT+'.oasive_quant_sandbox.g1_freddie_full61_'+suffix+'_pool_metadata',
              source_identities={s:core.table_id(CLIENT.get_table(s)) for s in sources},
              feature_start=max('2021-05-01',pd.to_datetime(target.issue_date).min().strftime('%Y-%m-01')),
              helper_sha256=sha(full.__file__),roster_csv_sha256=sha(ROOT/'tools/freddie_research_pool_roster.csv'),
              scope=CONTRACT['scope'],official_DPR_membership_certified=False)
    core.write_json(prep_file,PREP)
assert PREP['helper_sha256']==sha(full.__file__) and PREP['roster_csv_sha256']==sha(ROOT/'tools/freddie_research_pool_roster.csv')
roster_record=EVAL/'roster_table_identity.json'
if roster_record.exists():
    assert core.table_id(CLIENT.get_table(PREP['roster']))==json.loads(roster_record.read_text())
else:
    from datetime import datetime,timedelta,timezone
    upload=POOL[['pool_id','pool_coupon','actual_coupon','issue_date','pool_vintage','mapping_status']].copy()
    upload['issue_date']=pd.to_datetime(upload.issue_date).dt.date
    upload['pool_vintage']=pd.to_numeric(upload.pool_vintage).astype('Int64')
    schema=[bigquery.SchemaField('pool_id','STRING'),bigquery.SchemaField('pool_coupon','FLOAT64'),bigquery.SchemaField('actual_coupon','FLOAT64'),
            bigquery.SchemaField('issue_date','DATE'),bigquery.SchemaField('pool_vintage','INT64'),bigquery.SchemaField('mapping_status','STRING')]
    table=bigquery.Table(PREP['roster'],schema=schema);table.expires=datetime.now(timezone.utc)+timedelta(days=30)
    CLIENT.create_table(table)
    CLIENT.load_table_from_dataframe(upload,PREP['roster'],job_config=bigquery.LoadJobConfig(schema=schema,write_disposition='WRITE_EMPTY')).result()
    core.write_json(roster_record,core.table_id(CLIENT.get_table(PREP['roster'])))
print('Research roster:',PREP['roster'])


Known target pools: 12130 Earliest issue: 2024-01-01
Unresolved candidate pools: 53098
Research roster: gen-lang-client-0343560978.oasive_quant_sandbox.g1_freddie_full61_fdeaf114c8ee_pool_metadata


## Step 11 — Dry-run calendar census, mapping audit and freeze
Review scan estimates before Step12/13. Native61-month calendar remains May2021–May2026 even though target pools first exist in2024. Feature partitions can start at their earliest issue without dropping calendar rows. All data queries are versioned and resumable.


In [ ]:
QUERIES={'membership_census':full.census_sql(PREP['roster']),'mapping_audit':full.mapping_audit_sql(PREP['roster']),
         'freeze_panel':full.freeze_sql(PREP['roster'],PREP['panel'],PREP['feature_start'])}
SQL=EVAL/'sql';SQL.mkdir(exist_ok=True)
ESTIMATES=[]
for name,q in QUERIES.items():
    (SQL/(name+'.sql')).write_text(q)
    dry=expansion.dry_run(CLIENT,q,full.CAP);ESTIMATES.append(dict(query=name,**dry))
ESTIMATES=pd.DataFrame(ESTIMATES);ESTIMATES.to_csv(EVAL/'preparation_estimates.csv',index=False)
assert ESTIMATES.estimated_bytes.sum()+TRAIN_ESTIMATES.estimated_bytes.sum()<=full.TOTAL_CAP
print(ESTIMATES.to_string(index=False))


            query  estimated_bytes  estimated_gib                                                       sql_sha256
membership_census      30117072994      28.048710 0fb7a377986a0fe61cd6d40d4531989fa7256e00cd3160ff227fe4e3f000674d
    mapping_audit      30116318274      28.048007 d2795a4966d92802547b345c23a473cad9dac23507a45ba64873403300e7445e
     freeze_panel     313829872938     292.276845 3f9566c0be955ac92c4dc7bc768d9c5a1be828de90b0a22793bf3500f1dc078f


## Step 12 — Execute census and unresolved-mapping audit
122 month/coupon cells are required. Preserve every calendar month. Unknown-vintage premium pools may or may not belong to G1; their exposure is not a certified missing-G1 percentage. Review it before interpreting the selected subset as representative.


In [ ]:
expansion.assert_sources(CLIENT,PREP['source_identities'])
CENSUS=full.cached_query(CLIENT,'membership_census',QUERIES['membership_census'],EVAL)
CENSUS['feature_date']=pd.to_datetime(CENSUS.feature_date).dt.strftime('%Y-%m-%d')
assert len(CENSUS)==122 and CENSUS.feature_date.nunique()==61
CENSUS.to_csv(EVAL/'membership_calendar_122_cells.csv',index=False)
MAPPING=full.cached_query(CLIENT,'mapping_audit',QUERIES['mapping_audit'],EVAL)
MAPPING.to_csv(EVAL/'mapping_coverage_by_month.csv',index=False)
display(CENSUS);display(MAPPING)


membership_census job: b50f7f54-be0f-407e-a4a1-3d68d0ca90f3
mapping_audit job: 88f8d90b-3fb7-4c87-b112-a5d222eb5dcf


,feature_date,pool_coupon,issued_target_pools,member_rows,distinct_member_loans,member_upb
0,2021-05-01,6.0,0,0,0,0.000000e+00
1,2021-05-01,6.5,0,0,0,0.000000e+00
2,2021-06-01,6.0,0,0,0,0.000000e+00
3,2021-06-01,6.5,0,0,0,0.000000e+00
4,2021-07-01,6.0,0,0,0,0.000000e+00
...,...,...,...,...,...,...
117,2026-03-01,6.5,4967,90084,90084,2.576338e+10
118,2026-04-01,6.0,7163,164322,164322,4.879156e+10
119,2026-04-01,6.5,4967,87071,87071,2.468620e+10
120,2026-05-01,6.0,7163,161131,161131,4.761025e+10


,feature_date,verified_target_rows,verified_target_upb,uncertain_premium_rows,uncertain_premium_upb
0,2021-05-01,0,0.000000e+00,0,0.000000e+00
1,2021-06-01,0,0.000000e+00,0,0.000000e+00
2,2021-07-01,0,0.000000e+00,0,0.000000e+00
3,2021-08-01,0,0.000000e+00,0,0.000000e+00
4,2021-09-01,0,0.000000e+00,0,0.000000e+00
...,...,...,...,...,...
56,2026-01-01,270700,8.083514e+10,345743,1.156222e+11
57,2026-02-01,265677,7.888006e+10,338887,1.125749e+11
58,2026-03-01,259110,7.636502e+10,329821,1.085898e+11
59,2026-04-01,251393,7.347776e+10,318956,1.039348e+11


## Step 13 — Create one frozen monthly panel in the sandbox
Actual large query; no CREATE OR REPLACE and no production writes. This joins Freddie servicing and Freddie loan statics, the fixed Freddie pool roster, all25 features and verified payoff records. It does not filter to March survivors or require a previous-month record.


In [ ]:
record=EVAL/'frozen_panel_identity.json';job_record=EVAL/'freeze_job.json'
if record.exists():
    PANEL_ID=json.loads(record.read_text());assert core.table_id(CLIENT.get_table(PREP['panel']))==PANEL_ID
else:
    expansion.assert_sources(CLIENT,PREP['source_identities'])
    q=QUERIES['freeze_panel'];digest=hashlib.sha256(q.encode()).hexdigest()
    if job_record.exists():
        previous=json.loads(job_record.read_text());assert previous['sql_sha256']==digest
        job=CLIENT.get_job(previous['job_id'],location='us-central1')
    else:
        job=CLIENT.query(q,location='us-central1',job_config=bigquery.QueryJobConfig(maximum_bytes_billed=full.CAP))
        core.write_json(job_record,dict(job_id=job.job_id,sql_sha256=digest))
    print('Freeze job:',job.job_id);job.result()
    expansion.assert_sources(CLIENT,PREP['source_identities'])
    PANEL_ID=core.table_id(CLIENT.get_table(PREP['panel']));core.write_json(record,PANEL_ID)
print('Frozen Freddie panel:',PREP['panel'])


Freeze job: 4521dba3-60fd-478a-aa0f-3c46378c79d1
Frozen Freddie panel: gen-lang-client-0343560978.oasive_quant_sandbox.g1_freddie_full61_fdeaf114c8ee_panel


In [ ]:
from google.colab import files

uploaded = files.upload()
name = "freddie_step14_january_coverage_fix_20261102.py"
assert name in uploaded, f"Upload {name}"
exec(compile(uploaded[name], name, "exec"), globals())

Saving freddie_step14_january_coverage_fix_20261102.py to freddie_step14_january_coverage_fix_20261102.py
Reviewed January source gap recorded. Previous code/spec preserved in: /content/g1_freddie_full61_20261102/evaluation/january_source_gap_review_20261102T051946026436
Rerun Step 14, then continue Step 15 onward. Do not rerun fitting or the freeze.


## Step 14 — Validate the frozen panel
Counts/UPB must match the independent census, with unique keys and no genuine missing features/labels or definition differences. Null curtailment training eligibility is reported, not used to remove a valid payoff outcome. A large zero-event month is flagged for outcome review and excluded from headline summaries, avoiding the September endpoint problem in the Fannie follow-up.


In [ ]:
q=full.panel_audit_sql(PREP['panel']);expansion.dry_run(CLIENT,q,full.READ_CAP)
PANEL_AUDIT=full.cached_query(CLIENT,'panel_audit',q,EVAL,limit=full.READ_CAP)
PANEL_AUDIT.to_csv(EVAL/'panel_audit_raw.csv',index=False)
INVENTORY=full.validate_inventory(CENSUS,PANEL_AUDIT)
INVENTORY.to_csv(EVAL/'verified_inventory_122_cells.csv',index=False)
display(INVENTORY)
display(INVENTORY.loc[INVENTORY.large_zero_event_warning])


,feature_date,pool_coupon,issued_target_pools,member_rows,distinct_member_loans,member_upb,panel_rows,distinct_panel_loans,panel_upb,payoff_events,missing_features,missing_labels,training_rule_ineligible_rows,label_definition_gaps,balance_conflicts,large_zero_event_warning,status
0,2021-05-01,6.0,0,0,0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,False,no_issued_g1_pools
1,2021-05-01,6.5,0,0,0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,False,no_issued_g1_pools
2,2021-06-01,6.0,0,0,0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,False,no_issued_g1_pools
3,2021-06-01,6.5,0,0,0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,False,no_issued_g1_pools
4,2021-07-01,6.0,0,0,0,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,False,no_issued_g1_pools
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,2026-03-01,6.5,4967,90084,90084,2.576338e+10,90084.0,90084.0,2.576338e+10,3013.0,0.0,0.0,0.0,0.0,0.0,False,scored_retrospective_unconfirmed
118,2026-04-01,6.0,7163,164322,164322,4.879156e+10,164322.0,164322.0,4.879156e+10,3191.0,0.0,0.0,0.0,0.0,0.0,False,scored_retrospective_unconfirmed
119,2026-04-01,6.5,4967,87071,87071,2.468620e+10,87071.0,87071.0,2.468620e+10,2272.0,0.0,0.0,0.0,0.0,0.0,False,scored_retrospective_unconfirmed
120,2026-05-01,6.0,7163,161131,161131,4.761025e+10,161131.0,161131.0,4.761025e+10,2377.0,0.0,0.0,0.0,0.0,0.0,False,scored_retrospective_unconfirmed


,feature_date,pool_coupon,issued_target_pools,member_rows,distinct_member_loans,member_upb,panel_rows,distinct_panel_loans,panel_upb,payoff_events,missing_features,missing_labels,training_rule_ineligible_rows,label_definition_gaps,balance_conflicts,large_zero_event_warning,status


## Step 15 — Lock the original-versus-expanded scoring design
Primary probability scale is the same as the Fannie historical test: both trees receive the0.12 sampling odds correction, without an extra June adjustment. Freddie's registered0.1185 payoff factor is on RAW odds, not on already0.12-corrected odds; do not apply both factors. June calibration is outside the historical fitting/comparison window and should be tested separately on later months, not mixed into these headline results.


In [ ]:
SCORING=[]
for month in full.MONTHS:
    if INVENTORY.loc[INVENTORY.feature_date.eq(month),'member_rows'].sum()==0:continue
    for role,q in zip(['input','predict'],full.month_sql(PREP['panel'],month)):
        (SQL/(role+'_'+month+'.sql')).write_text(q)
        dry=expansion.dry_run(CLIENT,q,full.READ_CAP);SCORING.append(dict(month=month,role=role,**dry))
SCORING=pd.DataFrame(SCORING);SCORING.to_csv(EVAL/'scoring_estimates.csv',index=False)
assert TRAIN_ESTIMATES.estimated_bytes.sum()+ESTIMATES.estimated_bytes.sum()+SCORING.estimated_bytes.sum()<=full.TOTAL_CAP
ASSETS={'model':str(FIT_DIR/'freddie_expanded_25features.ubj'),'encoder':str(FIT_DIR/'fixed_freddie_encoder.joblib'),'module':str(ORIG/'post_rally_matched_test.py')}
SPEC=dict(version='freddie_full61_v1',months=full.MONTHS,panel=PREP['panel'],panel_identity=PANEL_ID,
          production_identity=PROD_ID,helper_sha256=sha(full.__file__),core_sha256=sha(core.__file__),asset_paths=ASSETS,
          asset_sha256={k:sha(v) for k,v in ASSETS.items()},training_start='2021-05-01',training_last_month='2026-05-01',
          scope=PREP['scope'],official_DPR_membership_certified=False,prior_month_required=False,
          probability_scale='both raw tree odds multiplied by .12; no extra calibration',
          registered_freddie_raw_payoff_factor=.1185,registered_fannie_factor_not_used=.1174,
          comparison_type='shared pooled production payoff tree vs agency-specific local expanded Freddie payoff tree',
          split_only_causal_test=False,independent_future_test=False,training_plan_sha256=sha(EXP_WORK/'training_plan.json'))
lock=EVAL/'run_spec.json'
if lock.exists():assert json.loads(lock.read_text())==SPEC,'Changed scoring design: use new run directory'
else:core.write_json(lock,SPEC)
print(SCORING.to_string(index=False))


     month    role  estimated_bytes  estimated_gib                                                       sql_sha256
2024-02-01   input          7314788       0.006812 4c0610f3927be3ca1c81e008ab77baa948350785c6886e5d7311cc86e981a1b5
2024-02-01 predict          7818891       0.007282 621b201ded437eb7ec232154b228f58c1677d50615c2032a72e39f02b373a0ad
2024-03-01   input         15706416       0.014628 17784c2d7c5b29ad73221ac8e852962d850d66072220dd2b2792aba54454ac68
2024-03-01 predict         15077809       0.014042 755ca8f6620b1d41221007ef0cff924ef15aa6665f0be2953336bfd89ab4de3f
2024-04-01   input         24867490       0.023160 dde23d09c8f536d7e91f4ecf50274cadc926272c5823cde7d053edbafcff32d7
2024-04-01 predict         23001411       0.021422 c9af4ee395e42424bb90e1ec20d0980dd15323975c30ec761a745edb3bb183d1
2024-05-01   input         37815478       0.035218 224ee8020bcaedb4e899f5a7c85f67bd0c21733d5b4317988efb00edd48d3c98
2024-05-01 predict         34175710       0.031829 b7acd86bc5ab148a67e70

## Step 16 — Preview March, then score every calendar month
No archived Freddie March accuracy result is invented. The preview is the first new result, checked against the independently frozen census. Positive gains mean lower loss for the expanded tree. Counts/weights/features are identical in each arm. Predict once per month and cache checksummed results; both curves use the actual same loans.


In [ ]:
preview=full.score_month(CLIENT,EVAL,SPEC,'2026-03-01',INVENTORY,EXPANDED,ENCODER,m,full.cached_query)
display(preview.loc[preview.grouping.isin(['overall','coupon'])])


input_2026-03-01 job: 50f3e0ae-a5df-4ca4-bc4c-48d28de3a96f
predict_2026-03-01 job: 9c6b0ec4-c759-43f4-8e0a-855aa8679d14


,month,stage,grouping,group,loans,payoff_events,upb,actual_payoff_pct,effective_loans,small_group_attention,...,production_logloss,expanded_predicted_pct,expanded_bias_pp,expanded_brier,expanded_logloss,abs_bias_gain_pp,brier_gain,logloss_gain,agency,status
0,2026-03-01,before,overall,all,259110,7717,7.636502e+10,3.588913,193019.127804,False,...,0.153863,3.306127,-0.282786,0.034140,0.148343,1.170991,0.000377,0.005520,freddie,scored_retrospective_unconfirmed
1,2026-03-01,before,coupon,6.0,169026,4704,5.060164e+10,3.377882,127309.846503,False,...,0.147708,3.146016,-0.231866,0.032198,0.141288,1.290109,0.000412,0.006420,freddie,scored_retrospective_unconfirmed
83,2026-03-01,before,coupon,6.5,90084,3013,2.576338e+10,4.003397,65717.464039,False,...,0.165953,3.620600,-0.382797,0.037952,0.162199,0.937032,0.000307,0.003754,freddie,scored_retrospective_unconfirmed


In [20]:
for month in full.MONTHS:
    report=full.score_month(CLIENT,EVAL,SPEC,month,INVENTORY,EXPANDED,ENCODER,m,full.cached_query)
    row=report.loc[report.grouping.eq('overall')].iloc[0]
    print(month,'loans:',int(row.loans),'status:',row.status)
RESULTS,PERIODS=full.collect(EVAL)


2021-05-01 loans: 0 status: no_issued_g1_pools
2021-06-01 loans: 0 status: no_issued_g1_pools
2021-07-01 loans: 0 status: no_issued_g1_pools
2021-08-01 loans: 0 status: no_issued_g1_pools
2021-09-01 loans: 0 status: no_issued_g1_pools
2021-10-01 loans: 0 status: no_issued_g1_pools
2021-11-01 loans: 0 status: no_issued_g1_pools
2021-12-01 loans: 0 status: no_issued_g1_pools
2022-01-01 loans: 0 status: no_issued_g1_pools
2022-02-01 loans: 0 status: no_issued_g1_pools
2022-03-01 loans: 0 status: no_issued_g1_pools
2022-04-01 loans: 0 status: no_issued_g1_pools
2022-05-01 loans: 0 status: no_issued_g1_pools
2022-06-01 loans: 0 status: no_issued_g1_pools
2022-07-01 loans: 0 status: no_issued_g1_pools
2022-08-01 loans: 0 status: no_issued_g1_pools
2022-09-01 loans: 0 status: no_issued_g1_pools
2022-10-01 loans: 0 status: no_issued_g1_pools
2022-11-01 loans: 0 status: no_issued_g1_pools
2022-12-01 loans: 0 status: no_issued_g1_pools
2023-01-01 loans: 0 status: no_issued_g1_pools
2023-02-01 lo

/content/g1_freddie_full61_20261102/tools/g1_freddie_full61.py:210: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return f"""SELECT a.as_of_date feature_date,


predict_2024-03-01 job: 8d023a8d-a8a0-4930-bb47-3b1b6ca14d38
2024-03-01 loans: 42735 status: scored_retrospective_unconfirmed
input_2024-04-01 job: 2d948222-0d01-42df-a2bc-b565d483d3e7


/content/g1_freddie_full61_20261102/tools/g1_freddie_full61.py:210: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return f"""SELECT a.as_of_date feature_date,


predict_2024-04-01 job: 33320ec5-1ba4-4355-a460-62986919f48b
2024-04-01 loans: 66977 status: scored_retrospective_unconfirmed
input_2024-05-01 job: e5c3417d-1191-439f-acf9-f9d56cb535ab


/content/g1_freddie_full61_20261102/tools/g1_freddie_full61.py:210: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return f"""SELECT a.as_of_date feature_date,


... (3 outputs left out)

/content/g1_freddie_full61_20261102/tools/g1_freddie_full61.py:210: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return f"""SELECT a.as_of_date feature_date,


predict_2024-07-01 job: 381c06b0-3a8e-4d16-9cad-b8f11369a83c
2024-07-01 loans: 162429 status: scored_retrospective_unconfirmed
input_2024-08-01 job: d7d92f56-3d26-4ea4-bdde-35b3409de48e
predict_2024-08-01 job: c85404d0-50c9-43af-b20c-03af0fe0b61b
2024-08-01 loans: 207287 status: scored_retrospective_unconfirmed
input_2024-09-01 job: e62515a8-b21b-437e-905f-29d27dc57030
predict_2024-09-01 job: 8a82041b-0c06-4771-9bb5-2fe00c5c9012
2024-09-01 loans: 238817 status: scored_retrospective_unconfirmed
input_2024-10-01 job: 5844bbdb-d697-45dc-adcb-093e44a7a03f


/content/g1_freddie_full61_20261102/tools/g1_freddie_full61.py:210: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return f"""SELECT a.as_of_date feature_date,


predict_2024-10-01 job: a545deb4-9fac-4d08-8a49-a46531411617
2024-10-01 loans: 276647 status: scored_retrospective_unconfirmed
input_2024-11-01 job: 78200c18-2673-4d31-b686-41d576e17858
predict_2024-11-01 job: e2304139-4a2b-4673-a44a-ca2426833d37
2024-11-01 loans: 291690 status: scored_retrospective_unconfirmed
input_2024-12-01 job: 974250e3-8a05-4b1a-a042-b70b78b1d495
predict_2024-12-01 job: 56747d5e-7583-48a8-afdc-666e0b825d53
2024-12-01 loans: 301959 status: scored_retrospective_unconfirmed
input_2025-01-01 job: 3cfc405f-7661-4414-b11f-ab74e6081e8a
predict_2025-01-01 job: 4a4ffdb5-5b3f-4e99-a088-75983148414b
2025-01-01 loans: 308099 status: scored_retrospective_unconfirmed
input_2025-02-01 job: e9abd119-6c76-4bc8-bd84-04c79ef44325
predict_2025-02-01 job: addf2ae2-aa21-4bf8-8d90-073e1e5f3575
2025-02-01 loans: 317588 status: scored_retrospective_unconfirmed
input_2025-03-01 job: ff61e802-a918-4150-9b16-63970842269e
predict_2025-03-01 job: 6b6e762b-c771-4784-9e2e-64e2b09e920b
2025-03-0

## Step 17 — Interpret the full calendar and controlled groups
Report all three losses, per coupon as well as combined. Overall errors can cancel across coupons. The two arms score exactly the same loans, so loan characteristics are controlled in the paired comparison; age/balance/status/state/property/occupancy/channel/FICO/DTI/LTV groups reveal where the gain or harm lies. Those subgroup differences are descriptive, not causal effects of balance or state.

`native_freddie_window` keeps61 calendar months in the main calendar file; summary errors use populated, non-warning months only. `common_fannie_overlap` uses dates from January2022 for comparison with Fannie. If both summaries have the same populated months because these pools did not yet exist in2021, that is expected, not independent evidence of two separate successes.

Only strong consistent improvements across coupon-level aggregate error AND Brier/log loss, without unacceptable small/delinquent regressions, support continuing. A March/rally gain alone does not show future skill. No confidence intervals or new seed replicates are claimed here.


In [21]:
cols=['month','group','loans','payoff_events','actual_payoff_pct','production_predicted_pct','expanded_predicted_pct',
      'production_bias_pp','expanded_bias_pp','abs_bias_gain_pp','brier_gain','logloss_gain','status']
display(RESULTS.loc[RESULTS.grouping.eq('overall'),cols])
display(PERIODS.loc[PERIODS.grouping.isin(['overall','coupon','small_all_ages'])])
display(PERIODS.loc[PERIODS.grouping.eq('status')&PERIODS['group'].str.endswith('_delinquent')])
print('Read monthly_age_balance.csv and the state/property/occupancy/channel/FICO/DTI/LTV rows in all_61_month_groups.csv.')


,month,group,loans,payoff_events,actual_payoff_pct,production_predicted_pct,expanded_predicted_pct,production_bias_pp,expanded_bias_pp,abs_bias_gain_pp,brier_gain,logloss_gain,status
0,2021-05-01,all,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_issued_g1_pools
1,2021-06-01,all,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_issued_g1_pools
2,2021-07-01,all,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_issued_g1_pools
3,2021-08-01,all,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_issued_g1_pools
4,2021-09-01,all,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no_issued_g1_pools
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5655,2026-01-01,all,270700,5023.0,2.222033,1.976910,2.411094,-0.245123,0.189061,0.056062,0.000044,0.000910,scored_retrospective_unconfirmed
5915,2026-02-01,all,265677,6567.0,3.004048,2.068816,2.931641,-0.935231,-0.072407,0.862825,0.000210,0.003272,scored_retrospective_unconfirmed
6175,2026-03-01,all,259110,7717.0,3.588913,2.135136,3.306127,-1.453777,-0.282786,1.170991,0.000377,0.005520,scored_retrospective_unconfirmed
6429,2026-04-01,all,251393,5463.0,2.482488,2.170536,2.544643,-0.311952,0.062155,0.249798,0.000027,0.000526,scored_retrospective_unconfirmed


,period,grouping,group,months,loan_months,upb_month_exposure_bn,outcome_status,production_weighted_abs_bias_pp,expanded_weighted_abs_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
30,native_freddie_window,coupon,6.0,28,4276335,1334.023855,historical_unconfirmed,0.272883,0.114402,0.158480,0.000053,0.001023
31,native_freddie_window,coupon,6.5,28,2474552,747.019140,historical_unconfirmed,0.268323,0.149481,0.118842,0.000051,0.000940
136,native_freddie_window,overall,all,28,6750887,2081.042995,historical_unconfirmed,0.269049,0.113737,0.155312,0.000052,0.000993
147,native_freddie_window,small_all_ages,6.0_lt150k,28,641640,67.543175,historical_unconfirmed,0.133107,0.110493,0.022615,-0.000002,0.000128
148,native_freddie_window,small_all_ages,6.5_lt150k,28,433608,44.343893,historical_unconfirmed,0.188392,0.120386,0.068006,-0.000005,0.000126
291,common_fannie_overlap,coupon,6.0,28,4276335,1334.023855,historical_unconfirmed,0.272883,0.114402,0.158480,0.000053,0.001023
292,common_fannie_overlap,coupon,6.5,28,2474552,747.019140,historical_unconfirmed,0.268323,0.149481,0.118842,0.000051,0.000940
397,common_fannie_overlap,overall,all,28,6750887,2081.042995,historical_unconfirmed,0.269049,0.113737,0.155312,0.000052,0.000993
408,common_fannie_overlap,small_all_ages,6.0_lt150k,28,641640,67.543175,historical_unconfirmed,0.133107,0.110493,0.022615,-0.000002,0.000128
409,common_fannie_overlap,small_all_ages,6.5_lt150k,28,433608,44.343893,historical_unconfirmed,0.188392,0.120386,0.068006,-0.000005,0.000126


,period,grouping,group,months,loan_months,upb_month_exposure_bn,outcome_status,production_weighted_abs_bias_pp,expanded_weighted_abs_bias_pp,weighted_abs_bias_gain_pp,weighted_brier_gain,weighted_logloss_gain
258,native_freddie_window,status,6.0_delinquent,27,52019,15.390674,historical_unconfirmed,0.471025,0.587854,-0.116829,-0.000111,-0.001394
260,native_freddie_window,status,6.5_delinquent,27,43677,13.630912,historical_unconfirmed,0.491956,0.562352,-0.070396,-0.000381,-0.002771
519,common_fannie_overlap,status,6.0_delinquent,27,52019,15.390674,historical_unconfirmed,0.471025,0.587854,-0.116829,-0.000111,-0.001394
521,common_fannie_overlap,status,6.5_delinquent,27,43677,13.630912,historical_unconfirmed,0.491956,0.562352,-0.070396,-0.000381,-0.002771
779,rally_oct_apr,status,6.0_delinquent,7,22608,6.663651,historical_unconfirmed,0.530551,0.724719,-0.194168,-0.000352,-0.003677
781,rally_oct_apr,status,6.5_delinquent,7,18650,5.796962,historical_unconfirmed,0.344270,0.458094,-0.113824,-0.000610,-0.005001


Read monthly_age_balance.csv and the state/property/occupancy/channel/FICO/DTI/LTV rows in all_61_month_groups.csv.


## Step 18 — Export and download BOTH checkpoints
Run after useful progress, including after fitting and again at completion. FULL includes saved models, exact code, training/source audits, training extraction shards and completed monthly inputs/predictions. SMALL has CSV/code/evidence but not model/data recovery. Keep both in `Y:/OASIVE`; runtime files disappear. Sandbox roster/panel expire after30days; backups do not extend expiration.


In [22]:
from datetime import datetime,timezone
EXPORT_DIR=Path('/content/g1_freddie_full61_downloads');EXPORT_DIR.mkdir(exist_ok=True)
assert not EXPORT_DIR.resolve().is_relative_to(ROOT.resolve())
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'_'+uuid.uuid4().hex[:8]
BACKUPS={}
for is_full in [False,True]:
    kind='FULL_recovery' if is_full else 'small_results';destination=EXPORT_DIR/(ROOT.name+'_'+stamp+'_'+kind+'.zip');hashes={}
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if not p.is_file() or p.name.endswith('.recovering'):continue
            if not is_full and p.suffix in ['.parquet','.ubj','.joblib']:continue
            name=p.relative_to(ROOT).as_posix();z.write(p,name);hashes[name]=sha(p)
        z.writestr('freddie_recovery_hashes.json',json.dumps(hashes,indent=2))
    BACKUPS[kind]=destination;print(kind,destination,'bytes',destination.stat().st_size,'sha256',sha(destination))


small_results /content/g1_freddie_full61_downloads/g1_freddie_full61_20261102_20261102T065244_c17762d1_small_results.zip bytes 3021343 sha256 7b3cd227774c09da48414d35700374d2f9826f502899e0693868d18c3a8818f5
FULL_recovery /content/g1_freddie_full61_downloads/g1_freddie_full61_20261102_20261102T065244_c17762d1_FULL_recovery.zip bytes 656849168 sha256 f10124b7a78df22ef4d30e994b4d9c768a43c72ccc09e93ab251667584c0662c


In [25]:
from google.colab import files
files.download(str(BACKUPS['small_results']))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from google.colab import files
files.download(str(BACKUPS['FULL_recovery']))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Resume, limitations and next phase
- Restore this notebook's own FULL checkpoint in Step1 and rerun checked setup. Cached source SELECTs/shards/months are reused; never rewrite a changed identity/spec to bypass checks.
- If Step9 has completed, skip the memory/load-heavy part of Step8 and let Step9 verify/reload the saved fit. Do not refit on later months to make a failed July/August test pass.
- Full risk-set evaluation is not the small training sample. No loan-hash sample filter is applied to the evaluation panel.
- Metadata unresolved premium rows are reported in `mapping_coverage_by_month.csv`. Exact official DPR membership is NOT certified. Obtain a dated official mapping before claiming that exact alert universe is fully covered. Do not infer missing issuance WALA from current WALA.
- Owner surface keys and the research actual-pool roster have different derivations. Keep the scope label in reports; do not call this final served-surface CPR accuracy. The G1 alert is voluntary pool CPR; this test uses full monthly payoff probabilities including buyouts.
- To test the actual SERVED Freddie surface, keep its October version, Freddie-specific calibration, curtailment head/cbar, surface axes, story rules and September anchor. Build a candidate curtailment/surface counterpart under the same recipe, and compare voluntary components with settled DPR data on identical pools/dates. Do not subtract a monthly payoff result from pool CPR or treat a September-anchored surface as a July forecast.
- The main native window ends May2026. After it passes, a later-month Freddie test should keep the newly saved expanded tree fixed, use June calibration only as a separate sensitivity, score July/August, and admit September only after settled outcomes are verified. The preceding Fannie export had all-zero September labels; never reuse those as certified truth.
- A split-only attribution test would also hold pooled agency mix, engine, sampling realization, encoder and stopping settings fixed on a common frame. This production-versus-candidate test answers practical candidate accuracy, not that narrower causal question.

## Outputs
Training: `source_audit.csv`, `source_support.csv`, `fit_validation_population_counts.csv`, `target_profile_state_property_DTI_support.csv`, `fit_audit.csv`, `original_baseline_replay.json`, `freddie_expanded_25features.ubj`, `fixed_freddie_encoder.joblib`, `fit_complete.json`.
Evaluation: `membership_calendar_122_cells.csv`, `mapping_coverage_by_month.csv`, `verified_inventory_122_cells.csv`, `all_61_calendar_months.csv`, `period_summary.csv`, `monthly_coupon.csv`, `monthly_small.csv`, `monthly_delinquent.csv`, `monthly_age_balance.csv`, `all_61_month_groups.csv`, `completion.json`, per-month paired predictions/category coverage.

The original owner lineage and source metadata used to design this procedure are included in the inputs package. The new Freddie model and monthly results have NOT been fitted/executed by preparing this notebook.
